# Tutorial 7: Translating Sentences It Has Never Seen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/07-real-translations.ipynb)

**Reading. About 30 minutes.**

Tutorials 3 and 4 trained on toy corpora and then translated sentences from that same
corpus, so neither model ever translated something it had not memorized. This one loads
a model trained on 64,311 real English-Spanish sentence pairs, the one whose attention
tutorial 5 reads, and points it at talks it has never seen.

**A warning before you start: the translations are not good.** That is the honest
result of this much data and a few minutes of training, and seeing exactly *how* it is
not good is the point.

By the end you will be able to:

1. say why a test set must be held out by whole talks, not by random sentences, and what a leaked one does to the score;
2. report a BLEU that someone else could compare with theirs: the score, how many sentences it was measured on, and its signature;
3. compare greedy decoding with beam search on an uncertain model, with everything else held fixed, and say what beam search fixes and what it cannot;
4. recognise an undertrained model from its output.

These serve Lecture 10's outcome *explain greedy against beam search and why a comparison must
pin decoding*.

**Running in Google Colab?** Run the next two cells: the first installs TorchLingo, the second
downloads the pretrained model and the corpus this tutorial reads, about 32 MB. There is
nothing to uncomment.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 10 (*Overview of MT Quality Estimation*). A head start on **A11**.

In [1]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [2]:
from pathlib import Path

from torchlingo.colab import setup

# The pretrained model and the corpus are in the repository, not the pip package; setup
# downloads whichever are missing.
env = setup(data=[
    "data/pretrained/model.pt",
    "data/pretrained/spm.model",
    "data/pretrained/test.tsv",
    "data/example.tsv",
])

import pandas as pd
import torch

from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.inference import beam_search_decode, greedy_decode
from torchlingo.models import SimpleTransformer

PRETRAINED = Path("data/pretrained")
device = torch.device("cpu")

checkpoint = torch.load(PRETRAINED / "model.pt", map_location=device, weights_only=False)
vocab = SentencePieceVocab(str(PRETRAINED / "spm.model"))

model = SimpleTransformer(
    src_vocab_size=checkpoint["vocab_size"],
    tgt_vocab_size=checkpoint["vocab_size"],
    **checkpoint["model_config"],
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"trained on {checkpoint['train_pairs']:,} sentence pairs "
      f"for {checkpoint['trained_minutes']} minutes")
print(f"final validation loss: {checkpoint['val_losses'][-1]:.3f}")
print(f"vocabulary: {len(vocab):,} subword pieces")

TorchLingo 0.2.6 (editable checkout) | PyTorch 2.13.0 | device cpu | not in Colab


  data/pretrained/model.pt    10.2 MB
  data/pretrained/spm.model     0.3 MB
  data/pretrained/test.tsv     0.3 MB
  data/example.tsv    21.0 MB
trained on 64,311 sentence pairs for 24.3 minutes
final validation loss: 3.695
vocabulary: 3,000 subword pieces


## The held-out set is genuinely held out

`data/pretrained/test.tsv` comes from **talks the model never saw** — not random
sentences pulled out of talks it trained on.

That distinction does most of the work here. Consecutive sentences in a
transcript share a speaker, a topic and a vocabulary, so holding out random
*sentences* leaves the model with the surrounding context of every one of them.
Scores taken that way look much better than the model deserves.

Holding out whole talks means every sentence below is about a subject, in a
voice, the model has never encountered.

In [3]:
test = pd.read_csv(PRETRAINED / "test.tsv", sep="\t", dtype=str, keep_default_na=False)
print(f"{len(test):,} held-out sentence pairs\n")


def translate(sentence, beam_size=None):
    """Translate one sentence, greedy by default."""
    ids = vocab.encode(sentence, add_special_tokens=True)
    src = torch.tensor([ids]).to(device)
    if beam_size is None:
        tokens = greedy_decode(model, src, max_len=60)[0]
    else:
        tokens = beam_search_decode(model, src, beam_size=beam_size, max_len=60)
    return vocab.decode(tokens, skip_special_tokens=True)


sample = test.sample(6, random_state=0)
for _, row in sample.iterrows():
    print(f"EN    {row['src']}")
    print(f"MODEL {translate(row['src'])}")
    print(f"REF   {row['tgt']}")
    print()

1,889 held-out sentence pairs

EN    It's more like, say, if you were a dog.


MODEL Es más, si uno se les gusta, se les gusta mucho más.
REF   Es como, digamos, si fueras un perro.

EN    This is where I come from, Todmorden.
MODEL Este es donde estoy viendo de dónde viene.
REF   Yo vengo de Todmorden.

EN    Because commercial companies are inherently designed to get as much out of you [as] they can get away with.


MODEL Porque los diseñadores están diseñados con los diseñadores de diseño de diseño de diseño de diseño.
REF   Porque las empresas están pensadas en esencia para conseguir lo máximo que puedan de nosotros.

EN    So if all human life depends on plants, doesn't it make sense that perhaps we should try to save them?
MODEL Así que si hay que hacer la vida, ¿no? ¿Podemos hacer que la vida en la vida?
REF   Así que si toda la vida humana depende de las plantas, ¿acaso no tendría sentido que intentáramos salvarlas?

EN    BG: Somebody in the audience asked me, How does he breathe up there?
MODEL BG: Algunos: ¿Cómo me preguntó en el auditivo?
REF   BG: Alguien de la audiencia me preguntó ¿Cómo hace para respirar allá arriba?,

EN    So the biggest and the most important thing is our neutral point-of-view policy.
MODEL Así que la mayor parte más importante es la mayor parte más importante y nuestra mayor parte de nuestra mutica.
REF   Lo más importante es nuestra política del punto de vista n

## Read those carefully

**It is producing Spanish.** Word order, agreement, function words, inverted
question marks. Nothing told it what Spanish *is*; it learned that from
examples. Look at the fifth pair: it even kept the `BG:` speaker tag, because
TED transcripts are full of them.

**And then look at what goes wrong.** The dominant failure is not invention, it
is *repetition*:

```
los diseñadores están diseñados con los diseñadores de diseño de diseño de diseño
la mayor parte más importante es la mayor parte más importante y nuestra mayor parte
Es más, si uno se les gusta, se les gusta mucho más.
```

This is the classic degenerate mode of an undertrained sequence model, and it
has a mechanical explanation. At every step the decoder picks the most probable
next token given what it has already produced. When the model is unsure of the
content, the safest continuation is often the phrase it just emitted — that
phrase is, after all, demonstrably likely in this context. Greedy decoding has
no memory of having said it before and no way to prefer novelty.

Two things in this library exist partly because of that failure: **beam search**,
which can prefer a different path, and the **length penalty** `alpha`, which
stops the search from being rewarded for producing more of the same. You will
see beam search earn its keep further down.

Fluency and correctness are separate properties. This model is small enough that
you can watch them come apart.

## Putting a number on it

BLEU compares the model's output to the reference translation by n-gram overlap.
A model scored on sentences it trained on can reach 100 and mean nothing; this one is
measured on unseen talks, so the number means something.

It will be low — around **7**, where production systems on English-Spanish score in
the 40s. That is not a failure of the exercise. It is the measurement working: the
translations you just read *are* almost unusable, and BLEU says so.

In [4]:
from torchlingo.evaluation import compute_bleu

subset = test.sample(200, random_state=1)
hypotheses = [translate(s) for s in subset["src"]]
score = compute_bleu(hypotheses, list(subset["tgt"]))

# Report all three, always: the score, what it was measured on, and the signature.
# BLEU counts matching n-grams, so it depends on where the token boundaries fall --
# score the same translations with a different tokenizer and the number moves, with
# nothing in the number to warn you. The signature records how it was computed.
print(f"BLEU {score.score:.2f} on {len(subset)} held-out sentences")
print(score.signature)

BLEU 7.17 on 200 held-out sentences
nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0


## Greedy versus beam search, on a model that is actually uncertain

On tutorial 8's toy model the two give **the same translation for every sentence**,
because a model that has memorized its answers has nothing to be uncertain about.
Every beam follows the same path.

This model is uncertain about nearly everything, so the comparison finally has
something to show.

In [5]:
compare = test.sample(60, random_state=2)
greedy_out = [translate(s) for s in compare["src"]]
beam_out = [translate(s, beam_size=5) for s in compare["src"]]
refs = list(compare["tgt"])

differ = sum(g != b for g, b in zip(greedy_out, beam_out))
print(f"greedy BLEU  {compute_bleu(greedy_out, refs).score:.2f}")
print(f"beam-5 BLEU  {compute_bleu(beam_out, refs).score:.2f}")
print(f"different translations: {differ}/{len(compare)}")
print()

for src, g, b in list(zip(compare["src"], greedy_out, beam_out))[:5]:
    if g == b:
        continue
    print(f"EN      {src[:70]}")
    print(f"greedy  {g[:70]}")
    print(f"beam-5  {b[:70]}")
    print()

greedy BLEU  6.21
beam-5 BLEU  7.89
different translations: 56/60

EN      You could change the genes in principle.
greedy  Podrían cambiar el mundo en el genoma.
beam-5  Podrían cambiar los genes.

EN      It hooks up all parts of the brain.
greedy  Todos los peces de los cerebro.
beam-5  Es parte de todo el cerebro.

EN      So the best place for x-ray crystallography was at the Cavendish Labor
greedy  Así que la mejor parte de California de California de California de Ca
beam-5  Así que la mejora de California de California de California de Califor

EN      Across age, across income, across culture.
greedy  Un cultivo, una cultura, cultura, cultura.
beam-5  Una cultura, una cultura de cultura, cultura.

EN      How beautiful it is to lose ourselves in these little streets on the i
greedy  Sin embargo, es que estos pequeños nos perdieron en nuestros pequeños 
beam-5  ¿Cuánto es tan hermoso en estos pequeños hermosos.



**Why is that BLEU lower than the one above?** A different sample: 60 sentences here,
200 there, and BLEU on 60 short sentences moves by a point or more with the draw. A score
without its test set and size is not a number you can compare with anything. What *is*
meaningful is the difference between the two rows: same sentences, same model, only the
decoder changed. Measured properly, over five samples of 200, beam search is worth about
**+1.6 BLEU** here, with error bars in [Decoding](../concepts/decoding.md).

**What it fixes.** In the first pair greedy committed early to `el mundo`, a likely
phrase after "Podrían cambiar", and had to continue from there; beam search was still
carrying a hypothesis that had not, and it finished nearly at the reference. In the
second, greedy invents *peces* (fish) and beam search does not.

**What it cannot fix.** In the third and fourth pairs both decoders loop, on
`de California` and on `cultura`. When a repetition is the best continuation for every
hypothesis in the beam, a wider beam has nothing better to offer: beam search searches
what the model believes more thoroughly, and cannot search its way out of it.

## Your turn: train one yourself

Loading a checkpoint shows you the destination. Training shows you the road.

The cell below trains a model from scratch for a few hundred steps on a slice of
the same data. **It will be much worse than the one above** — that one had
thirty-six epochs over the whole corpus, this gets a fraction of one.

The point is to see what undertrained looks like, so you can recognise it. An
undertrained translation model does not produce nonsense; it produces the *most
common* output it can, over and over.

In [6]:
from functools import partial

from torch import optim
from torch.utils.data import DataLoader

from torchlingo.config import Config
from torchlingo.data_processing.batching import collate_fn
from torchlingo.data_processing.dataset import NMTDataset

corpus = pd.read_csv("data/example.tsv", sep="\t", dtype=str, keep_default_na=False)
corpus = corpus[corpus["kind"] == "transcript"].head(4000)
corpus[["src", "tgt"]].to_csv("data/quick_train.tsv", sep="\t", index=False)

cfg = Config(batch_size=32)
dataset = NMTDataset(Path("data/quick_train.tsv"), src_vocab=vocab, tgt_vocab=vocab,
                     max_length=60)
loader = DataLoader(dataset, batch_size=32, shuffle=True,
                    collate_fn=partial(collate_fn, pad_idx=vocab.pad_idx))

torch.manual_seed(0)
quick = SimpleTransformer(
    src_vocab_size=len(vocab), tgt_vocab_size=len(vocab),
    **checkpoint["model_config"],
).to(device)

import tempfile

from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

train_model(quick, loader, num_epochs=1, config=cfg,
            optimizer=optim.Adam(quick.parameters(), lr=3e-4),
            checkpointer=TrainingCheckpointer("quick", checkpoint_dir=tempfile.mkdtemp(), verbose=False))

quick.eval()
for sentence in list(test["src"].head(3)):
    ids = vocab.encode(sentence, add_special_tokens=True)
    out = greedy_decode(quick, torch.tensor([ids]).to(device), max_len=40)[0]
    print(f"EN         {sentence[:66]}")
    print(f"UNDERTRAINED {vocab.decode(out, skip_special_tokens=True)[:66]}")
    print()

Epoch 1 Step 100/125 | Train Loss: 8.0312


Epoch 1/1 | Train: 8.0126
EN         I want to help you re-perceive what philanthropy is, what it could
UNDERTRAINED inoental para esoano yanoaz eso what menien es de crearatur es de 



EN         I want to start with these word pairs here.
UNDERTRAINED inoental parauébo graué Sí sec beaut call compart lado 40 beaut be

EN         We all know which side of these we'd like to be on.
UNDERTRAINED dónde es es es es es es es es es es es es es es es es es es es es 



Compare that to the pretrained model's output earlier. The undertrained one has
learned the *shape* of the output — Spanish-looking tokens, plausible length —
without learning what any particular sentence means. It has found the safest
guess and is repeating it.

That is what the first few hundred steps of every translation model look like.

## What would make this better

In rough order of how much they would buy:

| | |
| --- | --- |
| **More training** | Thirty-six epochs in half an hour. Real systems train for days. |
| **More data** | 64k pairs is small. Production systems use tens of millions. |
| **A bigger model** | 2.5M parameters. Production models are 100-1000× larger. |
| **Better data** | TED transcripts are one narrow domain of spoken register. |

Notice that model size is third. It is the lever people reach for first and it is
rarely the binding constraint at this scale.

## What you saw

A model translating text it was never trained on, and exactly how far short of useful
it falls. Both halves matter: the mechanics work end to end, and the gap between "the
mechanics work" and "this is a translation system" is enormous.

## What's next

- [Tutorial 8: Inference and Beam Search](08-inference-and-beamsearch.ipynb) — greedy and beam search, written to be read
- Read [Decoding](../concepts/decoding.md) for what beam search and `alpha` actually buy, measured on this model
- Read [When It Fails](../concepts/when-it-fails.md) for diagnosing a model that is not working
- Read [Vocabulary](../concepts/vocabulary.md) for why this uses subwords, measured
- Retrain the checkpoint yourself with `python scripts/train_example_model.py`